[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qussai96/ProtAudit/blob/main/ProtAudit_Colab.ipynb)

# ProtAudit: score a protein FASTA

Upload a protein FASTA containing **1–100 proteins** and run the frozen ProtAudit ProtT5 model. Higher scores indicate a more protein-like sequence.

The notebook produces:

- a TSV file with one protein-likeness score per FASTA record;
- the frozen validation-threshold call for each protein;
- a score-band summary plot; and
- a ZIP archive containing both outputs.

**Runtime:** choose **Runtime → Change runtime type → GPU** before running. ProtT5 model weights are downloaded from Hugging Face during the first run, so setup can take several minutes.

In [ ]:
#@title 1. Install and load ProtAudit
import json
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This notebook is designed to run in Google Colab.") from exc

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.38,<5", "sentencepiece>=0.2", "matplotlib>=3.7"
], check=True)

REPOSITORY_URL = "https://github.com/qussai96/ProtAudit.git"
REPOSITORY_DIR = Path("/content/ProtAudit")
if not REPOSITORY_DIR.exists():
    subprocess.run([
        "git", "clone", "--quiet", "--depth", "1",
        REPOSITORY_URL, str(REPOSITORY_DIR)
    ], check=True)
elif (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run([
        "git", "-C", str(REPOSITORY_DIR), "pull", "--quiet", "--ff-only"
    ], check=True)

required = [
    REPOSITORY_DIR / "embed.py",
    REPOSITORY_DIR / "score.py",
    REPOSITORY_DIR / "models/manifest.json",
    REPOSITORY_DIR / "models/prott5.pt",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Incomplete ProtAudit checkout: " + ", ".join(missing))

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Choose Runtime → Change runtime type → GPU, "
        "then rerun the notebook from the beginning."
    )

manifest = json.loads((REPOSITORY_DIR / "models/manifest.json").read_text())
model_info = manifest["models"]["prott5"]
print("GPU:", torch.cuda.get_device_name(0))
print("ProtAudit repository loaded.")
print(f"Frozen ProtT5 threshold: {model_info['validation_threshold']:.8f}")

## Upload the FASTA

Requirements:

- upload exactly one FASTA file;
- identifiers (the first token after `>`) must be unique;
- sequences may contain the 20 standard amino acids or `X`; and
- the file must contain no more than 100 proteins.

Long proteins are processed using non-overlapping 1,000-residue windows and a residue-weighted mean; residues are not truncated.

In [ ]:
#@title 2. Upload one protein FASTA (maximum 100 proteins)
MAX_SEQUENCES = 100
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Please upload exactly one FASTA file.")

uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
INPUT_FASTA = Path("/content/protaudit_input.faa")
INPUT_FASTA.write_bytes(uploaded_bytes)

sys.path.insert(0, str(REPOSITORY_DIR))
from embed import read_fasta

records = list(read_fasta(INPUT_FASTA))
if not records:
    raise ValueError("The uploaded FASTA contains no protein records.")
if len(records) > MAX_SEQUENCES:
    raise ValueError(
        f"The uploaded FASTA contains {len(records)} proteins; this Colab is limited "
        f"to {MAX_SEQUENCES}. Use the local ProtAudit release for larger files: "
        f"{REPOSITORY_URL}"
    )

lengths = [len(sequence) for _, sequence in records]
print(f"Accepted {len(records)} protein(s) from {uploaded_name}")
print(f"Length range: {min(lengths):,}–{max(lengths):,} aa")
print(f"Total residues: {sum(lengths):,}")

## Generate ProtT5 embeddings and score proteins

The embedding step is the slow part. The code automatically splits a batch if the GPU runs out of memory.

In [ ]:
#@title 3. Generate ProtT5 embeddings
EMBEDDING_DIR = Path("/content/protaudit_embeddings")

command = [
    sys.executable, str(REPOSITORY_DIR / "embed.py"), str(INPUT_FASTA),
    "--output", str(EMBEDDING_DIR),
    "--model", "prott5",
    "--device", "cuda",
    "--max-batch-tokens", "4000",
    "--max-batch-size", "8",
    "--overwrite",
]
subprocess.run(command, check=True)

embedding_summary = json.loads((EMBEDDING_DIR / "summary.json").read_text())
if embedding_summary.get("record_count") != len(records):
    raise RuntimeError("Embedding row count does not match the uploaded FASTA.")
print("Embedding validation passed:", embedding_summary["embedding_shape"])

In [ ]:
#@title 4. Score, display, and plot the proteins
import pandas as pd
from IPython.display import Image, display

OUTPUT_DIR = Path("/content/ProtAudit_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCORE_TABLE = OUTPUT_DIR / "protaudit_scores.tsv"
SCORE_PLOT = OUTPUT_DIR / "protaudit_scores_plot.png"

subprocess.run([
    sys.executable, str(REPOSITORY_DIR / "score.py"), str(EMBEDDING_DIR),
    "--output", str(SCORE_TABLE),
    "--plot", str(SCORE_PLOT),
], check=True)

results = pd.read_csv(SCORE_TABLE, sep="\t")
if len(results) != len(records) or results["protein_id"].tolist() != [x[0] for x in records]:
    raise RuntimeError("Score-table identifiers do not match the uploaded FASTA.")

display(results)
display(Image(filename=str(SCORE_PLOT)))

In [ ]:
#@title 5. Download the scores and plot
import shutil

archive = Path(shutil.make_archive(
    "/content/ProtAudit_results", "zip", root_dir=OUTPUT_DIR
))
print(f"Downloading {archive.name} ({archive.stat().st_size / 1024:.1f} KiB)")
files.download(str(archive))

## Interpretation

- `protein_likeness_score` ranges from 0 to 1; higher scores are more protein-like.
- `passes_frozen_threshold` uses the ProtT5 threshold selected by maximum F1 on the frozen plant validation species (**0.45833838**). It was not fitted to the uploaded FASTA.
- `score_band` divides scores into `>=0.9`, `0.5–0.9`, and `<0.5` descriptive bands. These bands are not calibrated probabilities or separate trained thresholds.
- A low score prioritizes a model for inspection; it does not by itself identify the underlying genomic annotation error.
- For files larger than 100 proteins, use the [local ProtAudit release](https://github.com/qussai96/ProtAudit).